# 🚀 Bane Agent: High-Speed Neural DPO Benchmark (Google Colab T4 GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pariveshkoshta-spec/Bane_Agent/blob/main/Bane_Colab_GPU_Benchmark.ipynb)

> **Optimized for Speed:** Zero slow recursive drive scans, instant library caching (0.01s), and direct path loading from Google Drive.

### Step 1: Instant Dependency Check (~10s first time, 0.01s after)

In [ ]:
import torch
assert torch.cuda.is_available(), "❌ GPU NOT ACTIVE! In Colab top menu: Runtime -> Change runtime type -> select 'T4 GPU' -> Save"
print(f"✅ GPU Active: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB VRAM)")

# Check if already installed to skip downloading
try:
    import peft
    import bitsandbytes
    import faiss
    import sentence_transformers
    print("⚡ All libraries already installed! Skipped download in 0.01s.")
except ImportError:
    print("⏳ First-time install of lightweight inference packages (~10s)...")
    !pip install -q peft bitsandbytes faiss-cpu sentence-transformers rich tabulate
    print("✅ Setup complete!")

### Step 2: Clone Bane Agent & Ensure Database Ready (~3s)

In [ ]:
import os
if not os.path.exists("/content/Bane_Agent") and not os.path.exists("Bane_Agent"):
    !git clone -q https://github.com/pariveshkoshta-spec/Bane_Agent.git

if os.path.exists("/content/Bane_Agent"):
    %cd -q /content/Bane_Agent
elif os.path.exists("Bane_Agent"):
    %cd -q Bane_Agent

!git pull -q

# Ensure database exists
if not os.path.exists("enterprise_nexus.sqlite"):
    !python scripts/generate_enterprise_nexus.py

print(f"✅ Enterprise Database Ready: {os.path.exists('enterprise_nexus.sqlite')}")

### Step 3: Fast Direct Google Drive Mount (~2s, Zero Lag) 📂

In [ ]:
from google.colab import drive
import os
import shutil
import zipfile

# 1. Mount Drive
if not os.path.exists('/content/drive/MyDrive'):
    print("⏳ Mounting Google Drive... please allow access in the popup:")
    drive.mount('/content/drive')
print("✅ Google Drive connected!")

local_dir = "/content/local_adapters"

# 2. Fast check: If already staged from a previous run, skip entirely!
if os.path.exists(os.path.join(local_dir, "adapter_config.json")):
    print(f"⚡ Adapters already cached locally at: {local_dir} (Ready in 0.01s)!")
    adapter_path = local_dir
else:
    # Fast Direct Check: Folder in Drive (Instant O(1), no slow recursive scanning)
    direct_folders = [
        "/content/drive/MyDrive/bane_dpo_lora_adapters",
        "/content/drive/MyDrive/bane_dpo_lora_adapters/results/bane_dpo_lora_adapters",
        "/content/drive/MyDrive/results/bane_dpo_lora_adapters",
        "/content/drive/MyDrive/Bane_Agent/results/bane_dpo_lora_adapters",
        "/content/drive/MyDrive/Colab Notebooks/bane_dpo_lora_adapters"
    ]
    
    found_folder = None
    for cand in direct_folders:
        if os.path.exists(os.path.join(cand, "adapter_config.json")):
            found_folder = cand
            break
            
    if found_folder:
        print(f"📁 Detected adapter folder: {found_folder}")
        print("⚡ Quick-copying to high-speed local SSD (~2s)... ")
        os.makedirs(local_dir, exist_ok=True)
        for item in os.listdir(found_folder):
            s = os.path.join(found_folder, item)
            d = os.path.join(local_dir, item)
            if os.path.isfile(s):
                shutil.copy2(s, d)
        print(f"✅ Adapters staged to SSD: {local_dir}")
        adapter_path = local_dir
    else:
        # Fast Direct Check: Zip file in Drive root (Instant)
        direct_zips = [
            "/content/drive/MyDrive/bane_dpo_lora_adapters.zip",
            "/content/drive/MyDrive/adapters.zip",
            "/content/drive/MyDrive/results.zip"
        ]
        found_zip = None
        for zp in direct_zips:
            if os.path.exists(zp):
                found_zip = zp
                break
                
        if found_zip:
            print(f"📦 Detected adapter zip: {found_zip}. Extracting directly to SSD (~3s)...")
            os.makedirs(local_dir, exist_ok=True)
            temp_extract = "/content/temp_extract"
            with zipfile.ZipFile(found_zip, 'r') as zf:
                zf.extractall(temp_extract)
            # Find folder containing adapter_config.json
            extracted_target = temp_extract
            for r, dirs, files in os.walk(temp_extract):
                if "adapter_config.json" in files:
                    extracted_target = r
                    break
            for item in os.listdir(extracted_target):
                s = os.path.join(extracted_target, item)
                d = os.path.join(local_dir, item)
                if os.path.isfile(s):
                    shutil.copy2(s, d)
            print(f"✅ Extracted and staged to SSD: {local_dir}")
            adapter_path = local_dir
        else:
            adapter_path = None
            print("❌ Could not find 'bane_dpo_lora_adapters' folder or zip in your Google Drive root.")
            print("Contents of your Google Drive root:")
            for f in os.listdir('/content/drive/MyDrive')[:15]:
                print(f"  • {f}")

### Step 4: Load Base Model + LoRA Weights into 16GB GPU VRAM (~15s)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

assert adapter_path is not None and os.path.exists(os.path.join(adapter_path, "adapter_config.json")), \
    f"❌ Please check Step 3: adapter_config.json was not found!"

base_model_id = "unsloth/llama-3-8b-instruct-bnb-4bit"
print(f"⏳ Loading 4-bit base model ({base_model_id}) on T4 GPU...")

tokenizer = AutoTokenizer.from_pretrained(adapter_path)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("⏳ Attaching fine-tuned DPO LoRA adapters...")
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()
print(f"🚀 Model + LoRA weights successfully loaded into GPU VRAM!")

### Step 5: Run Full 30-Question Benchmark at GPU Speed (~1s per query) 🚀

In [ ]:
import sqlite3
import time
import json
from src.db_introspector import DatabaseIntrospector
from src.schema_rag import SchemaRetriever
from src.prompt_builder import format_dpo_prompt
from scripts.evaluate_part_a import test_suite_part_a
from scripts.evaluate_part_b import part_b_questions

# 1. Initialize FAISS schema index
introspector = DatabaseIntrospector("enterprise_nexus.sqlite")
schemas = introspector.extract_schemas()
retriever = SchemaRetriever()
retriever.build_index(schemas)
print(f"✅ FAISS indexed {len(schemas)} enterprise tables into vector space.\n")

# 2. Prepare 30 benchmark queries
all_questions = []
for q in test_suite_part_a:
    all_questions.append({
        "id": q["id"],
        "title": q["title"],
        "question": q["question"],
        "expected_sql": q["expected_sql"],
        "type": "Part A (Technical)"
    })
for q in part_b_questions:
    all_questions.append({
        "id": q["id"],
        "title": q["title"],
        "question": q["question"],
        "expected_sql": q["expected_sql"],
        "type": "Part B (Conversational)"
    })

db_conn = sqlite3.connect("enterprise_nexus.sqlite")
cursor = db_conn.cursor()
results = []

print(f"🔥 Evaluating {len(all_questions)} Queries on NVIDIA T4 GPU...\n" + "="*75)

for item in all_questions:
    q_id = item["id"]
    q_text = item["question"]
    exp_sql = item["expected_sql"].strip()

    # RAG schema retrieval
    context = retriever.retrieve_context(q_text, top_k=3)
    prompt = format_dpo_prompt(q_text, context)

    # Neural inference on GPU
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    t0 = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    latency = time.time() - t0
    raw_sql = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    gen_sql = raw_sql.replace(prompt, "").strip()

    # Clean markdown formatting
    if "```sql" in gen_sql:
        gen_sql = gen_sql.split("```sql")[1].split("```")[0].strip()
    elif "```" in gen_sql:
        gen_sql = gen_sql.split("```")[1].split("```")[0].strip()

    # Execute on enterprise_nexus.sqlite
    exec_status = "SUCCESS"
    err_str = None
    gen_rows = []
    try:
        cursor.execute(gen_sql)
        gen_rows = cursor.fetchall()
    except Exception as e:
        exec_status = "EXEC_ERROR"
        err_str = str(e)

    # Execute expected
    cursor.execute(exp_sql)
    exp_rows = cursor.fetchall()

    icon = "✅" if exec_status == "SUCCESS" else "❌"
    print(f"[{icon}] Q{q_id:02d} [{item['type']}]: {item['title']} ({latency:.2f}s | {len(gen_rows)} rows)")
    if exec_status != "SUCCESS":
        print(f"      Error: {err_str}")
        print(f"      SQL:   {gen_sql[:100]}...")

    results.append({
        "id": q_id,
        "title": item["title"],
        "type": item["type"],
        "question": q_text,
        "agent_sql": gen_sql,
        "expected_sql": exp_sql,
        "status": exec_status,
        "error": err_str,
        "gen_rows": len(gen_rows),
        "exp_rows": len(exp_rows),
        "latency": round(latency, 2)
    })

db_conn.close()

# Save results
with open("colab_gpu_results.json", "w") as f:
    json.dump(results, f, indent=2)

total = len(results)
pass_count = sum(1 for r in results if r["status"] == "SUCCESS")
print("\n" + "="*75)
print(f"🎉 Complete! Pass Rate: {pass_count}/{total} ({pass_count/total*100:.1f}%)")

### Step 6: Render Comprehensive Scorecard & Query Review Table

In [ ]:
from IPython.display import display, Markdown

part_a_succ = sum(1 for r in results if "Part A" in r["type"] and r["status"] == "SUCCESS")
part_b_succ = sum(1 for r in results if "Part B" in r["type"] and r["status"] == "SUCCESS")

md_scorecard = f"""
## 📊 GPU Neural Benchmark Scorecard

| Section | Total Queries | Syntax Execution Pass | Accuracy Rate |
| :--- | :---: | :---: | :---: |
| **Part A: Technical & Analytical** | 20 | **{part_a_succ}/20** | **{part_a_succ/20*100:.1f}%** |
| **Part B: Conversational Slang** | 10 | **{part_b_succ}/10** | **{part_b_succ/10*100:.1f}%** |
| **TOTAL OVERALL** | **30** | **{pass_count}/30** | **{pass_count/30*100:.1f}%** |
"""
display(Markdown(md_scorecard))

# Show sample query outputs
print("\nSample Query Outputs:")
for r in results[:5]:
    display(Markdown(f"### Q{r['id']}: {r['title']}\n**Question:** *{r['question']}*\n\n**Generated SQL:**\n```sql\n{r['agent_sql']}\n```\n**Target SQL:**\n```sql\n{r['expected_sql']}\n```"))